In [28]:
from open_clip import create_model_and_transforms  # 新增：open_clip加载
from torch.optim import Adam
from torch.nn import functional as F
from datasets import load_from_disk
import torch.nn as nn
from PIL import Image
import torch
import torch.nn.functional as F  # for cosine if needed

# 加载本地safetensors
safetensors_path = '/root/BriLLM0.5/CLIP-ViT-bigG-14-laion2B-39B-b160k/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors'

# 用open_clip create bigG模型 (ViT-bigG-14)
model, _, preprocess = create_model_and_transforms(
    'ViT-bigG-14', pretrained=None  # None to load custom checkpoint
)
model.load_state_dict(torch.load(safetensors_path, map_location='cpu'), strict=True)  # 加载safetensors
model = model.visual  # 只取vision部分
model = model.cuda().eval()

# cond_proj (1536 for bigG -> 32)
cond_proj = nn.Linear(1536, 32).cuda()
optimizer = Adam(cond_proj.parameters(), lr=1e-3)

ds = load_from_disk('coco_subset')
for epoch in range(3):
    total_loss = 0
    num_batches = 0
    for i in range(0, len(ds), 4):  # mini-batch=4
        batch = ds[i:i+4]
        images = [batch['image'][j].convert('RGB') for j in range(len(batch['image']))]  # ds PIL
        inputs = preprocess(images).unsqueeze(0).to('cuda')  # open_clip transforms (b,3,224,224)
        vision_out = model(inputs)  # (4,1536) CLS token or mean-pool if needed
        vision_out = vision_out.mean(1) if vision_out.dim() > 2 else vision_out  # ensure (4,1536)
        cond = cond_proj(vision_out)  # (4,32)
        target = torch.randn(4, 32).cuda()  # 自监督简化
        loss = F.mse_loss(cond, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    print(f'Epoch {epoch}: Avg loss = {total_loss / num_batches:.4f}')
torch.save(cond_proj.state_dict(), 'cond_proj.pth')
print('Saved bigG projector.')

UnpicklingError: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: Unsupported operand 72

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.